In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns



In [ ]:
# SHARED CHART STYLE (applied once, used by every chart below)
# A small, consistent palette instead of seaborn's rotating defaults --
# each color has a fixed MEANING across every chart in this script, so
# a reader learns the color code once and can read every chart in the
# deck without re-reading a legend each time.
PALETTE = {
    "primary": "#2E5EAA",    # neutral / "this is the process" blue
    "accent": "#D64550",     # problem / claim-vs-actual gap / violation
    "good": "#3AA655",       # on-target / cleaned-up state
    "warn": "#F2A93B",       # caution / secondary comparison
    "muted": "#9AA0A6",      # background / "before" state
}
sns.set_theme(style="whitegrid", font_scale=1.05)
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.edgecolor": "#4A4A4A",
    "axes.titleweight": "bold",
    "axes.titlesize": 13,
    "axes.titlepad": 12,
    "axes.labelsize": 11,
    "font.size": 10.5,
    "grid.color": "#E3E3E3",
    "grid.linewidth": 0.6,
    "savefig.dpi": 200,
    "savefig.bbox": "tight",
})
 
 
def label_bars(ax, fmt="%.1f", padding=3, fontsize=9.5):
    """Print each bar's value above/beside it, so the chart is
    self-contained without the reader needing the printed table too."""
    for container in ax.containers:
        ax.bar_label(container, fmt=fmt, padding=padding, fontsize=fontsize,
                      color="#2B2B2B", fontweight="medium")
 
 
def style_axes(ax, title, subtitle=None, ylabel=None, xlabel=None):
    """Apply a consistent title/subtitle/spine treatment to one axes.
    The title is pushed up with extra pad so the subtitle (a separate
    text object, since matplotlib only supports one real title per
    axes) has clear room directly beneath it without overlapping."""
    ax.set_title(title, loc="left", pad=26 if subtitle else 12)
    if subtitle:
        ax.text(0, 1.015, subtitle, transform=ax.transAxes,
                 fontsize=9.5, color="#6B6B6B", va="bottom")
    if ylabel is not None:
        ax.set_ylabel(ylabel)
    if xlabel is not None:
        ax.set_xlabel(xlabel)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

In [ ]:
# STEP 1 — LOAD DATA
# The 'Orders' sheet has a blank first row before the real header, hence header=1.
# The other 3 sheets have headers on row 0 (default).

orders = pd.read_excel("data/Muesli_Project_raw_data.xlsx", sheet_name="Orders", header=1)
campaign = pd.read_excel("data/Muesli_Project_raw_data.xlsx", sheet_name="Campaign Data")
process = pd.read_excel("data/Muesli_Project_raw_data.xlsx", sheet_name="Order Process Data")
intern = pd.read_excel("data/Muesli_Project_raw_data.xlsx", sheet_name="InternData Study")

print("Raw shapes:")
print(f"  orders   : {orders.shape}")
print(f"  process  : {process.shape}")
print(f"  intern   : {intern.shape}")
print(f"  campaign : {campaign.shape}")

In [ ]:
orders

In [ ]:
# STEP 2 — INITIAL DATA QUALITY CHECK (missing values, dtypes, duplicates)

print("Orders — full-row duplicates:", orders.duplicated().sum())      #.sum() turns it into one whole count of how many duplicate rows exist total
print("Campaign Data — full-row duplicates:", campaign.duplicated().sum())
print("Order Process Data — full-row duplicates:", process.duplicated().sum())
print("InternData Study — full-row duplicates:", intern.duplicated().sum())

In [ ]:
# 1. Isolate all 86 duplicate rows (showing all copies)
duplicates_df = intern[intern.duplicated(keep=False)]
# 2. Sort by Order ID so the identical rows sit right next to each other
duplicates_df_sorted = duplicates_df.sort_values(by='Order ID')
# 3. View the first 10 rows to inspect them
duplicates_df_sorted.head(10)

In [ ]:
#Missing values

print("\n--- Missing values per sheet ---")
for name, df in [("orders", orders), ("process", process),
                  ("intern", intern), ("campaign", campaign)]:
    na_counts = df.isna().sum()
    print(f"{name}:\n{na_counts[na_counts > 0]}\n")
 
print("--- Duplicate Order IDs per sheet ---")
for name, df in [("orders", orders), ("process", process),
                  ("intern", intern), ("campaign", campaign)]:
    total_rows = len(df)
    unique_ids = df["Order ID"].nunique()
    full_row_dupes = df.duplicated().sum()
    if name == "process":
        full_row_dupes = df.iloc[:, 1:].duplicated().sum()  # exclude first column (Row ID)
    elif name == "orders":
            full_row_dupes = df.iloc[:, 1:].duplicated().sum()  # exclude first column (Row ID)
    else:
            full_row_dupes = df.duplicated().sum()
    print(f"{name}: {total_rows} rows, {unique_ids} unique Order IDs, "
          f"{full_row_dupes} exact duplicates")

In [ ]:

intern_clean = intern.drop_duplicates()
process_clean = process.drop_duplicates(subset=process.columns[1:])
 
# Sanity check: after removing exact copies, are there any REMAINING
# Order IDs with more than one row (i.e. a genuine conflict)?
remaining_conflicts = intern_clean[intern_clean["Order ID"].duplicated(keep=False)]
print(f"\nintern_clean: {intern_clean.shape[0]} rows, "
      f"{intern_clean['Order ID'].nunique()} unique orders, "
      f"{remaining_conflicts['Order ID'].nunique()} still-conflicting IDs after cleanup")
 
# Campaign data had no duplicates or missing values -- used as-is.
campaign_clean = campaign.drop(columns=["Customer Name"])  # not needed for timing analysis

In [ ]:
# --- Chart: before vs after row counts for the intern sheet ---
fig, ax = plt.subplots(figsize=(4.5, 4.5))
bars = ax.bar(["Before dedupe", "After dedupe"], [len(intern), len(intern_clean)],
               color=[PALETTE["muted"], PALETTE["good"]], width=0.55)
ax.bar_label(bars, fmt="%.0f", padding=4, fontsize=10.5, fontweight="bold")
pct_removed_i = 100 * (1 - len(intern_clean) / len(intern))
style_axes(ax, "InternData Study: row count",
           subtitle=f"{pct_removed_i:.0f}% were exact copy-paste duplicates",
           ylabel="Rows")
plt.tight_layout()
plt.savefig("chart_step04_intern_dedupe.png")
plt.close()
print("Saved chart_step04_intern_dedupe.png")

In [ ]:


missing_mask = orders["Postal Code"].isna()
print(f"Rows with missing Postal Code before fill: {missing_mask.sum()}")
print(orders.loc[missing_mask, ["City", "State"]].drop_duplicates())

orders.loc[
    (orders["City"] == "Burlington") & (orders["State"] == "Vermont"),
    "Postal Code"
] = 5401  # official USPS ZIP code for Burlington, VT

print(f"Rows with missing Postal Code after fill: {orders['Postal Code'].isna().sum()}")

In [ ]:
# --- Chart: before vs after row counts for the process sheet ---
before_after = pd.DataFrame({
    "stage": ["Before dedupe", "After dedupe"],
    "rows": [len(process), len(process_clean)],
})
fig, ax = plt.subplots(figsize=(4.5, 4.5))
bars = ax.bar(before_after["stage"], before_after["rows"],
               color=[PALETTE["muted"], PALETTE["good"]], width=0.55)
ax.bar_label(bars, fmt="%.0f", padding=4, fontsize=10.5, fontweight="bold")
pct_removed = 100 * (1 - len(process_clean) / len(process))
style_axes(ax, "Order Process Data: row count",
           subtitle=f"{pct_removed:.0f}% of rows were duplicate line items",
           ylabel="Rows")
plt.tight_layout()
plt.savefig("chart_step03_process_dedupe.png")
plt.close()
print("Saved chart_step03_process_dedupe.png")

In [ ]:
orders.info()

In [ ]:
# STEP 4 — BUILD THE MASTER ORDER-LEVEL TABLE
 
master = process_clean.merge(intern_clean, on="Order ID", how="left")
master = master.merge(campaign_clean, on="Order ID", how="left")
 
print(f"\nmaster table: {master.shape[0]} orders")
print("Coverage of the sparse columns (how many orders have each date):")
print(master[["Ready to Ship Date", "Pickup Date", "Arrival Scan Date"]].notna().sum())
 
 

In [ ]:
# --- Chart: data coverage across the sparse columns ---
# Why a bar chart: 3 sparse columns, simplest way to show "how much do we have" out of the full master table.
coverage = master[["Ready to Ship Date", "Pickup Date", "Arrival Scan Date"]].notna().sum()
coverage_pct = (coverage / len(master) * 100).round(1)
fig, ax = plt.subplots(figsize=(6.5, 4.2))
bars = ax.bar(coverage_pct.index, coverage_pct.values, color=PALETTE["primary"], width=0.5)
ax.bar_label(bars, labels=[f"{v}%\n(n={c})" for v, c in zip(coverage_pct.values, coverage.values)],
             padding=4, fontsize=9.5)
ax.set_ylim(0, max(coverage_pct.values) * 1.35)
style_axes(ax, "Data coverage of the sparse columns",
           subtitle=f"Out of {len(master)} total orders in the master table",
           ylabel="% of orders with a recorded date")
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
plt.xticks(rotation=10)
plt.tight_layout()
plt.savefig("chart_step05_data_coverage.png")
plt.close()
print("Saved chart_step05_data_coverage.png")

In [ ]:
master.head(10)

In [ ]:
# STEP 5 — COMPUTE STAGE DURATIONS (the core KPIs)
# Each KPI maps to one gap in the pipeline. All in calendar days.
 
master["processing_time"] = (master["Ready to Ship Date"] - master["Order Date"]).dt.days
master["ship_delay"]      = (master["On Truck Scan Date"] - master["Ready to Ship Date"]).dt.days
master["transit_time"]    = (master["Arrival Scan Date"] - master["On Truck Scan Date"]).dt.days
master["total_lead_time"] = (master["Arrival Scan Date"] - master["Order Date"]).dt.days

stage_cols = ["processing_time", "ship_delay", "transit_time", "total_lead_time"]
 
# Data quality check: negative durations would mean a scan was recorded
# BEFORE the previous stage -- almost certainly a data entry error. reporting them rather than silently dropping them.
neg_counts = {}
for col in stage_cols:
    n_negative = (master[col] < 0).sum()
    n_valid = master[col].notna().sum()
    neg_counts[col] = n_negative
    print(f"{col}: {n_negative} negative value(s) out of {n_valid} non-null")
 

In [ ]:
# --- Chart: negative-value counts per KPI (should all be zero) ---
fig, ax = plt.subplots(figsize=(6.5, 4))
bars = ax.bar(list(neg_counts.keys()), list(neg_counts.values()), color=PALETTE["good"], width=0.5)
label_bars(ax, fmt="%.0f")
ax.set_ylim(0, 1)
ax.set_yticks([0, 1])
style_axes(ax, "Negative (impossible) values per stage",
           subtitle="Confirms no scan was recorded before the previous stage",
           ylabel="Count")
plt.xticks(rotation=10)
plt.tight_layout()
plt.savefig("chart_step06_negative_values.png")
plt.close()
print("Saved chart_step06_negative_values.png")

In [ ]:
# STEP 6 — DESCRIPTIVE STATISTICS PER STAGE
# Mean alone hides risk -- always report spread (std) and the 95th percentile, since the business cares about worst-case reliability,
# not just the average case.
 
desc_stats = master[stage_cols].describe(percentiles=[.5, .75, .95]).round(2)
print("\n--- Descriptive statistics per stage (days) ---")
print(desc_stats)
 

In [ ]:
# --- Chart: distribution of each stage duration, with mean/median labeled ---
fig, axes = plt.subplots(2, 2, figsize=(12, 8.5))
for ax, col in zip(axes.flat, stage_cols):
    sns.histplot(master[col].dropna(), kde=True, ax=ax,
                 color=PALETTE["primary"], edgecolor="white", alpha=0.85)
    mean_v, med_v = master[col].mean(), master[col].median()
    ax.axvline(mean_v, color=PALETTE["accent"], linestyle="--", linewidth=1.6)
    ax.axvline(med_v, color=PALETTE["good"], linestyle="--", linewidth=1.6)
    ax.text(mean_v, ax.get_ylim()[1] * 0.92, f" mean {mean_v:.1f}",
            color=PALETTE["accent"], fontsize=9, fontweight="bold")
    ax.text(med_v, ax.get_ylim()[1] * 0.78, f" median {med_v:.1f}",
            color=PALETTE["good"], fontsize=9, fontweight="bold")
    style_axes(ax, col.replace("_", " ").title(), ylabel="Orders", xlabel="Days")
plt.tight_layout()
plt.savefig("chart_step07_stage_distributions.png")
plt.close()
print("Saved chart_step07_stage_distributions.png")

In [ ]:
# STEP 7 — VALIDATE THE WAREHOUSE MANAGER'S CLAIMS
# Claim 1: Processing normally takes 2 business days.
# Claim 2: Express orders ship the SAME day as ready (ship_delay == 0).
# Claim 3: Trucks only leave Monday / Wednesday / Friday.
  
# --- Claim 1: processing time by Ship Mode ---
print("\n--- Processing & ship delay, split by Ship Mode ---")
print(master.groupby("Ship Mode")[["processing_time", "ship_delay"]]
      .describe(percentiles=[.5, .95]).round(2))
 
# --- Claim 2: Express same-day shipping match rate ---
# Only evaluate rows where we actually HAVE ship_delay (i.e. intern data present)
express_with_data = master[(master["Ship Mode"] == "Express") & master["ship_delay"].notna()]
express_match_rate = (express_with_data["ship_delay"] == 0).mean()
print(f"\nExpress same-day-ship match rate: {express_match_rate:.1%} "
      f"(n={len(express_with_data)})")
 
# --- Claim 3: truck departure day-of-week rule ---
master["truck_dow"] = master["On Truck Scan Date"].dt.day_name()
valid_truck_days = {"Monday", "Wednesday", "Friday"}
truck_day_violation_rate = 1 - master["truck_dow"].isin(valid_truck_days).mean()
print(f"\nTruck-day-rule (Mon/Wed/Fri only) violation rate: "
      f"{truck_day_violation_rate:.1%} across all {len(master)} orders")
print(master["truck_dow"].value_counts(normalize=True).round(3))

In [ ]:
# --- Chart: processing_time & ship_delay by Ship Mode, with claim lines ---
fig, axes = plt.subplots(1, 2, figsize=(11, 5.2))
sns.boxplot(data=master, x="Ship Mode", y="processing_time", ax=axes[0],
            hue="Ship Mode", palette=[PALETTE["warn"], PALETTE["primary"]], legend=False,
            width=0.5, fliersize=3)
axes[0].axhline(2, color=PALETTE["accent"], linestyle="--", linewidth=1.5)
axes[0].text(0.98, 2, " claimed: 2 days", transform=axes[0].get_yaxis_transform(),
             ha="right", va="bottom", color=PALETTE["accent"], fontsize=9, fontweight="bold")
style_axes(axes[0], "Processing Time by Ship Mode", ylabel="Days")
 
sns.boxplot(data=master, x="Ship Mode", y="ship_delay", ax=axes[1],
            hue="Ship Mode", palette=[PALETTE["warn"], PALETTE["primary"]], legend=False,
            width=0.5, fliersize=3)
axes[1].axhline(0, color=PALETTE["accent"], linestyle="--", linewidth=1.5)
axes[1].text(0.98, 0, " Express promise: 0 days", transform=axes[1].get_yaxis_transform(),
             ha="right", va="bottom", color=PALETTE["accent"], fontsize=9, fontweight="bold")
style_axes(axes[1], "Ship Delay by Ship Mode",
           subtitle=f"Express hits same-day only {express_match_rate:.0%} of the time",
           ylabel="Days")
plt.tight_layout()
plt.savefig("chart_step08a_shipmode_boxplots.png")
plt.close()
print("Saved chart_step08a_shipmode_boxplots.png")
 
# --- Chart: truck departure day-of-week distribution ---
truck_day_counts = master["truck_dow"].value_counts().reindex(
    ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]
)
bar_colors = [PALETTE["primary"] if d in valid_truck_days else PALETTE["accent"]
              for d in truck_day_counts.index]
fig, ax = plt.subplots(figsize=(7.5, 4.5))
bars = ax.bar(truck_day_counts.index, truck_day_counts.values, color=bar_colors, width=0.55)
label_bars(ax, fmt="%.0f")
style_axes(ax, "Truck Departures by Weekday",
           subtitle=f"Red = rule violation ({truck_day_violation_rate:.1%} of all orders)",
           ylabel="Number of orders")
plt.tight_layout()
plt.savefig("chart_step08b_truck_day_violations.png")
plt.close()
print("Saved chart_step08b_truck_day_violations.png")

In [ ]:
# STEP 8 — COMPARE ACTUAL VS. CLAIMED AVERAGES (summary table)
# =====================================================================
comparison = pd.DataFrame({
    "Stage": ["Processing (Order->Ready)", "Ship delay (Ready->Truck)",
              "Transit (Truck->Arrival)", "Total lead time"],
    "Claimed (days)": [2, "1 (2 if no truck)", 3, "n/a"],
    "Actual mean (days)": [
        round(master["processing_time"].mean(), 2),
        round(master["ship_delay"].mean(), 2),
        round(master["transit_time"].mean(), 2),
        round(master["total_lead_time"].mean(), 2),
    ],
    "Actual 95th %ile (days)": [
        master["processing_time"].quantile(.95),
        master["ship_delay"].quantile(.95),
        master["transit_time"].quantile(.95),
        master["total_lead_time"].quantile(.95),
    ],
})
print("\n--- Claimed vs. Actual summary ---")
print(comparison.to_string(index=False))
 

In [ ]:
# --- Chart: claimed vs. actual, side by side, with a % gap label ---
plot_df = comparison[comparison["Stage"] != "Total lead time"].copy()
plot_df["Claimed (numeric)"] = [2, 1.5, 3]  # numeric stand-in for the mixed-type claim column
plot_df["gap_pct"] = ((plot_df["Actual mean (days)"] / plot_df["Claimed (numeric)"]) - 1) * 100
 
x = np.arange(len(plot_df))
width = 0.35
fig, ax = plt.subplots(figsize=(9, 5.5))
b1 = ax.bar(x - width / 2, plot_df["Claimed (numeric)"], width, label="Claimed", color=PALETTE["muted"])
b2 = ax.bar(x + width / 2, plot_df["Actual mean (days)"], width, label="Actual mean", color=PALETTE["accent"])
ax.bar_label(b1, fmt="%.1f", padding=3, fontsize=9.5)
ax.bar_label(b2, fmt="%.1f", padding=3, fontsize=9.5, fontweight="bold")
for xi, gap in zip(x, plot_df["gap_pct"]):
    ax.text(xi + width / 2, plot_df["Actual mean (days)"].iloc[xi] + 0.6,
            f"+{gap:.0f}%", ha="center", color=PALETTE["accent"], fontsize=9.5, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(plot_df["Stage"], rotation=10)
ax.legend(frameon=False, loc="upper left")
style_axes(ax, "Claimed vs. Actual Average Duration",
           subtitle="Every stage runs slower than the company's own estimate",
           ylabel="Days")
plt.tight_layout()
plt.savefig("chart_step09_claimed_vs_actual.png")
plt.close()
print("Saved chart_step09_claimed_vs_actual.png")
 

In [ ]:
# STEP 9 — DIAGNOSTIC CUTS (where is the time/variation coming from?)
# Trend over time: is the process getting worse or better?
master["order_month"] = master["Order Date"].dt.to_period("M")
monthly_trend = master.groupby("order_month")[stage_cols].mean().round(2)
print("\n--- Monthly average duration trend (first 6 months shown) ---")
print(monthly_trend.head(6))
 
# Reliability flag: stages where std dev is large relative to the mean
# suggest an unpredictable process -- worth calling out to the business.
reliability = pd.DataFrame({
    "mean": master[stage_cols].mean(),
    "std": master[stage_cols].std(),
})
reliability["coefficient_of_variation"] = (reliability["std"] / reliability["mean"]).round(2)
print("\n--- Reliability check (higher CoV = less predictable stage) ---")
print(reliability.round(2))
 

In [ ]:
# --- Chart: monthly trend line, larger markers, direct end-of-line labels ---
monthly_trend_plot = master.groupby("order_month")[stage_cols].mean()
monthly_trend_plot.index = monthly_trend_plot.index.astype(str)
line_colors = {"processing_time": PALETTE["primary"], "ship_delay": PALETTE["warn"],
               "transit_time": PALETTE["accent"], "total_lead_time": PALETTE["good"]}
fig, ax = plt.subplots(figsize=(11, 5.5))
for col in stage_cols:
    series = monthly_trend_plot[col].dropna()
    ax.plot(series.index, series.values, marker="o", markersize=5, linewidth=2,
            color=line_colors[col], label=col.replace("_", " "))
ax.legend(frameon=False, loc="upper left", ncol=2)
style_axes(ax, "Average Stage Duration by Order Month",
           subtitle="Gaps in a line = months with no intern/campaign data that month",
           ylabel="Days")
plt.xticks(rotation=60, fontsize=8)
plt.tight_layout()
plt.savefig("chart_step10a_monthly_trend.png")
plt.close()
print("Saved chart_step10a_monthly_trend.png")
 
# --- Chart: reliability (Coefficient of Variation), ranked ---
reliability_sorted = reliability.sort_values("coefficient_of_variation")
fig, ax = plt.subplots(figsize=(7.5, 4.5))
bars = ax.barh(reliability_sorted.index, reliability_sorted["coefficient_of_variation"],
               color=PALETTE["primary"], height=0.5)
label_bars(ax, fmt="%.2f")
style_axes(ax, "Reliability by Stage",
           subtitle="Higher = less predictable relative to its own average",
           xlabel="Coefficient of Variation (std / mean)")
plt.tight_layout()
plt.savefig("chart_step10b_reliability_cov.png")
plt.close()
print("Saved chart_step10b_reliability_cov.png")
 

In [ ]:
# STEP 10 — OUTLIER DETECTION

# Why the IQR method, not just eyeballing min/max: the mean/std already
# told us the shape is roughly symmetric but not perfectly normal, and
# these are day-count values (bounded at 0, right-skewed tails), so the
# IQR (Tukey) method is more robust here than a z-score cutoff, which
# assumes normality and gets distorted by the very outliers we're
# trying to find.

# We also check outliers WITHIN each Ship Mode group separately, not
# just on the pooled column. Reason: Express and Standard have
# genuinely different expected durations (a 5-day processing_time is
# normal for Standard but a major outlier for Express). Pooling them
# would hide Express-specific problems and flag normal Standard orders
# as false positives.
 
def iqr_outlier_bounds(series):
    """Return (lower_bound, upper_bound) using the 1.5*IQR Tukey rule."""
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr
 
 
print("\n--- Outlier detection (1.5x IQR rule), pooled across all orders ---")
outlier_summary = []
for col in stage_cols:
    s = master[col].dropna()
    lower, upper = iqr_outlier_bounds(s)
    n_outliers = ((s < lower) | (s > upper)).sum()
    outlier_summary.append({
        "stage": col,
        "lower_bound": round(lower, 2),
        "upper_bound": round(upper, 2),
        "n_outliers": n_outliers,
        "pct_of_non_null": round(100 * n_outliers / len(s), 1),
    })
outlier_summary_df = pd.DataFrame(outlier_summary)
print(outlier_summary_df.to_string(index=False))
 
# Same check, but split by Ship Mode -- catches group-specific issues
# that the pooled view above would mask.
print("\n--- Outlier detection, split by Ship Mode ---")
for col in ["processing_time", "ship_delay"]:  # the two stages Ship Mode affects
    print(f"\n{col}:")
    for mode, group in master.groupby("Ship Mode"):
        s = group[col].dropna()
        if len(s) == 0:
            continue
        lower, upper = iqr_outlier_bounds(s)
        n_outliers = ((s < lower) | (s > upper)).sum()
        pct = 100 * n_outliers / len(s)
        flag = ""
        if upper == lower:
            # Degenerate case: Q1 == Q3 means the middle 50% of values are
            # IDENTICAL (zero IQR), so the 1.5*IQR rule collapses to
            # "anything not equal to this exact value" -- not a meaningful
            # outlier test anymore. Flag it rather than report a
            # misleadingly high outlier count.
            flag = "  <-- IQR=0 here: most values are identical, so this " \
                   "rule isn't meaningful; treat as a distribution note, not true outliers"
        print(f"  {mode}: bounds=({lower:.2f}, {upper:.2f}), "
              f"{n_outliers} outlier(s) out of {len(s)} ({pct:.1f}%){flag}")
 
# Pull out the actual outlier rows for total_lead_time -- these are the
# specific orders a stakeholder would want investigated individually
# (e.g. "why did order X take 17 days end to end?").
lower_tlt, upper_tlt = iqr_outlier_bounds(master["total_lead_time"].dropna())
lead_time_outliers = master[
    (master["total_lead_time"] < lower_tlt) | (master["total_lead_time"] > upper_tlt)
][["Order ID", "Order Date", "Ship Mode", "total_lead_time"]].sort_values(
    "total_lead_time", ascending=False
)
print(f"\n--- Specific total_lead_time outlier orders (n={len(lead_time_outliers)}) ---")
print(lead_time_outliers.to_string(index=False))
 
# Negative durations (checked in Step 6) are a DIFFERENT kind of outlier:
# they're impossible values (data errors), not just extreme-but-real
# values. Keep that distinction in the write-up -- IQR outliers are
# "unusually slow/fast but plausible", negatives are "broken data".

In [ ]:
# --- Chart: outlier boxplots with median, n, and outlier % annotated per stage ---
fig, ax = plt.subplots(figsize=(9.5, 5.5))
box_colors = [PALETTE["primary"], PALETTE["warn"], PALETTE["accent"], PALETTE["good"]]
box_data = [master[c].dropna() for c in stage_cols]
bp = ax.boxplot(box_data, patch_artist=True,
                 tick_labels=[c.replace("_", " ") for c in stage_cols],
                 widths=0.5, flierprops=dict(marker="o", markersize=5,
                                              markerfacecolor=PALETTE["accent"],
                                              markeredgecolor="none", alpha=0.8))
for patch, color in zip(bp["boxes"], box_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.55)
for median in bp["medians"]:
    median.set_color("#2B2B2B")
    median.set_linewidth(1.6)
 
y_top = ax.get_ylim()[1]
for i, (row, series) in enumerate(zip(outlier_summary_df.itertuples(), box_data)):
    median_val = series.median()
    # Outlier % header, above each box
    ax.text(i + 1, y_top * 0.98, f"{row.pct_of_non_null:.0f}% outliers",
            ha="center", va="top", fontsize=9, color="#2B2B2B", fontweight="bold")
    # n, just below the header
    ax.text(i + 1, y_top * 0.91, f"n={len(series)}",
            ha="center", va="top", fontsize=8.5, color="#6B6B6B")
    # Median value, labeled just to the right of the median line itself
    ax.text(i + 1.32, median_val, f"median {median_val:.1f}",
            ha="left", va="center", fontsize=8.5, color="#2B2B2B",
            fontweight="medium")
 
style_axes(ax, "Outlier Detection per Stage",
           subtitle="Points beyond the whiskers = 1.5x IQR outliers",
           ylabel="Days")
plt.tight_layout()
plt.savefig("chart_step11_outlier_boxplots.png")
plt.close()
print("Saved chart_step11_outlier_boxplots.png")

In [ ]:
# STEP 11 — SAVE CLEANED MASTER TABLE FOR FURTHER USE (slides, extra credit, etc.)

master.to_csv("muesli_master_cleaned.csv", index=False)
print("\nSaved cleaned & merged dataset to muesli_master_cleaned.csv")